# Mission3 데이터 검토 — 학습 없이 오심 오류 찾기

이 노트북 하나만 Colab에 업로드하고 **런타임 유형은 Python 3 / 하드웨어 가속기 없음(CPU)** 으로 실행하세요.

기존 ver9 실행 때 사용한 Drive 자료를 재사용합니다.
- `내 드라이브/mission3_ver5_inputs.zip`
- `내 드라이브/DCC_AIdeas/results/mission3/ver9/mission3_ver9_model.ckpt`
- 같은 ver9 폴더의 `run_identity.json`

ZIP에서 서울 Training의 보정 집합 2,880건과 저장된 예측 확률만 읽습니다. 모델 재학습·GPU·가중치 로딩·외부 API 호출은 없습니다.
공식 Validation은 읽지 않습니다. **여기 나오는 숫자는 Training 내부 오류 집계이며 최종 평가 점수가 아닙니다.**

1. 아래 경로를 확인하고 모두 실행합니다. Drive 연결을 승인합니다.
2. 첫날은 오심 80건(FP/FN/TP/TN 각 20건)을 읽습니다. 자동 원인 판정은 하지 않습니다.
3. 검토 화면에서 전사를 읽고, 정답·예측 보기를 누른 뒤 분류와 근거를 적어 저장합니다.
4. 저장 버튼마다 CSV가 Drive에 저장됩니다. 같은 설정으로 재실행하면 이어서 검토합니다.
5. 두 사람이 따로 검토하려면 `REVIEWER`를 서로 다르게 설정하세요.

**FP:** 오심 정답이 없는데 예측함 · **FN:** 오심 정답을 놓침 · **TP/TN:** 맞힌 양성/음성 대조 사례.
표본을 유형별 같은 수로 뽑으므로 80건 전체의 비율을 모집단 오류율로 해석하지 않습니다.


In [ ]:
from pathlib import Path
INPUT_ZIP = "/content/drive/MyDrive/mission3_ver5_inputs.zip"
VER9_DIR = "/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver9"
OUTPUT_ROOT = "/content/drive/MyDrive/DCC_AIdeas/results/mission3/data_review"
REVIEWER = "joo"  # 다른 검토자는 yeun, inho 등 다른 영문 이름 사용
FOCUS = ["오심"]  # 다음 단계: ["오심", "두통", "전신쇠약"]
PER_GROUP = 20
SEED = 42


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [Path(INPUT_ZIP), Path(VER9_DIR)/"mission3_ver9_model.ckpt", Path(VER9_DIR)/"run_identity.json"]:
    if not path.is_file():
        raise FileNotFoundError(f"파일을 찾지 못했습니다. 설정 셀의 경로를 확인하세요: {path}")
print("필수 파일 확인 완료")


In [ ]:
"""CPU-only review of frozen ver9 baseline errors on Training calibration.

This module reads the original input ZIP and small JSON/NPZ artifacts. It does
not load model weights, train, tune thresholds, or read official Validation.
The companion Colab notebook embeds this source; uploading it alone is enough.
"""
import csv
import hashlib
import io
import json
import random
import re
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

LABELS = ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']
CATEGORIES = {
    'model_missed_evidence': '명확한 근거를 모델이 놓침',
    'question_negation_history': '질문·부정·과거 문맥을 혼동',
    'symptom_confusion': '다른 증상과 혼동한 것으로 의심',
    'text_label_unclear': '전사만으로 정답 근거를 판단하기 어려움',
    'correct_clear': '맞힌 사례이며 근거도 명확함',
    'other': '기타: 메모에 구체적으로 기록',
}
SCOPE = 'Training calibration; frozen fitted thresholds; descriptive, not independent evaluation'


def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha(data):
    return hashlib.sha256(data).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def load_analysis(input_zip, ver9_dir):
    ver9_dir = Path(ver9_dir)
    manifest = read_json(ver9_dir / 'mission3_ver9_model.ckpt')
    identity = read_json(ver9_dir / 'run_identity.json')
    require(manifest.get('format') == 'mission3_ver9' and manifest.get('labels') == LABELS,
            'ver9 manifest 또는 증상 순서가 다릅니다.')
    require(manifest.get('signature') == identity.get('signature'), '서로 다른 실행의 ver9 파일입니다.')
    selection = manifest['selection']
    require(manifest.get('selected_name') == 'ver4_baseline'
            and manifest.get('status') == 'selection_frozen'
            and selection.get('blend_space') == 'raw'
            and selection.get('candidate_weights') == [0.0] * 9,
            '이 노트북은 현재 ver9의 확정된 ver4 baseline 결과 전용입니다.')
    alpha = np.asarray(selection['neural_weights'], dtype=np.float64)
    thresholds = np.asarray(selection['thresholds'], dtype=np.float64)
    require(alpha.shape == thresholds.shape == (9,) and np.isfinite(alpha).all()
            and np.isfinite(thresholds).all() and ((alpha >= 0) & (alpha <= 1)).all(),
            '잘못된 혼합 비율 또는 임계값입니다.')

    with zipfile.ZipFile(input_zip) as archive:
        members = archive.namelist()
        require(len(members) == len(set(members)), 'ZIP 안에 중복 경로가 있습니다.')

        def unique(suffix):
            found = [name for name in members if name.endswith(suffix)]
            require(len(found) == 1, f'ZIP 안에 {suffix} 파일이 정확히 하나 있어야 합니다.')
            return found[0]

        cache_bytes = archive.read(unique('/ver4/calibration_predictions.npz'))
        split_bytes = archive.read(unique('/ver4/splits.json'))
        require(sha(cache_bytes) == identity.get('calibration_sha256'), 'ver9와 원본 예측 캐시의 해시가 다릅니다.')
        require(sha(split_bytes) == identity.get('splits_sha256'), 'ver9와 원본 데이터 분할의 해시가 다릅니다.')
        splits = json.loads(split_bytes)
        with np.load(io.BytesIO(cache_bytes), allow_pickle=False) as cache:
            names = cache['filenames'].tolist()
            truth = cache['targets'].copy()
            neural = cache['neural'].astype(np.float64)
            sparse = cache['sparse'].astype(np.float64)
        require(len(names) > 0 and len(names) == len(set(names)), '예측 파일명이 비어 있거나 중복됩니다.')
        require(len(splits['calibration']) == len(set(splits['calibration']))
                and set(names) == set(splits['calibration']), '보정 집합의 파일 목록이 다릅니다.')
        for fold in splits['folds']:
            require(not set(names).intersection(fold['fit'] + fold['stop']), '학습/stop과 보정 집합이 겹칩니다.')
        for arr in (truth, neural, sparse):
            require(arr.shape == (len(names), 9) and np.isfinite(arr).all()
                    and ((arr >= 0) & (arr <= 1)).all(), '잘못된 예측 배열입니다.')
        require(np.isin(truth, [0, 1]).all(), '정답은 0 또는 1이어야 합니다.')

        # Read only the original Training calibration JSONs, without extracting
        # the multi-GB model weights or opening any Validation member.
        training = {}
        for name in members:
            if '/TL_서울_구급/' in name and name.endswith('.json'):
                basename = name.rsplit('/', 1)[-1]
                require(basename not in training, 'Training에 중복 파일명이 있습니다.')
                training[basename] = name
        require(set(names) <= set(training), 'ZIP에 필요한 서울 Training JSON이 없습니다.')
        texts = []
        for i, name in enumerate(names):
            raw = json.loads(archive.read(training[name]).decode('utf-8-sig'))
            symptoms = raw['symptom']
            if symptoms is None:
                symptoms = []
            if isinstance(symptoms, str):
                symptoms = [symptoms]
            require(isinstance(symptoms, list) and all(isinstance(s, str) for s in symptoms),
                    f'정답 형식 오류: {name}')
            symptoms = {s.strip() for s in symptoms}
            require(np.array_equal([int(s in symptoms) for s in LABELS], truth[i]),
                    f'원본과 캐시의 정답 불일치: {name}')
            utterances = raw.get('utterances', raw.get('utterences'))
            require(isinstance(utterances, list), f'전사 형식 오류: {name}')
            turns = []
            for turn in utterances:
                if isinstance(turn, dict) and isinstance(turn.get('text'), str):
                    text = ' '.join(unicodedata.normalize('NFC', turn['text']).split())
                    if text:
                        turns.append(text)
            texts.append('\n'.join(turns))
            if (i + 1) % 500 == 0 or i + 1 == len(names):
                print(f'Training 전사 확인: {i + 1:,}/{len(names):,}', flush=True)
    probabilities = alpha * neural + (1 - alpha) * sparse
    text_digest = sha(json.dumps(list(zip(names, texts)), ensure_ascii=False).encode())
    return dict(names=names, truth=truth, texts=texts, probabilities=probabilities,
                predictions=probabilities >= thresholds, thresholds=thresholds,
                cache_sha256=sha(cache_bytes), text_sha256=text_digest,
                signature=manifest['signature'], scope=SCOPE)


def confusion_rows(data):
    rows = []
    for j, symptom in enumerate(LABELS):
        y, p = data['truth'][:, j].astype(bool), data['predictions'][:, j]
        tp, fp = int((y & p).sum()), int((~y & p).sum())
        fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())
        rows.append(dict(symptom=symptom, TP=tp, FP=fp, FN=fn, TN=tn,
                         precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                         f1=2 * tp / max(2 * tp + fp + fn, 1)))
    return rows


def make_samples(data, focus=('오심',), per_group=20, seed=42):
    require(type(per_group) is int and per_group > 0, '표본 수는 양의 정수여야 합니다.')
    require(len(focus) > 0 and len(focus) == len(set(focus)) and set(focus) <= set(LABELS), '증상 설정을 확인하세요.')
    records = []
    for symptom in focus:
        j = LABELS.index(symptom)
        y, p = data['truth'][:, j].astype(bool), data['predictions'][:, j]
        masks = [('FP', ~y & p), ('FN', y & ~p), ('TP', y & p), ('TN', ~y & ~p)]
        for group_index, (kind, mask) in enumerate(masks):
            indices = sorted(np.flatnonzero(mask).tolist(), key=lambda i: data['names'][i])
            rng = random.Random(seed + j * 1009 + group_index)
            for i in rng.sample(indices, min(per_group, len(indices))):
                records.append(dict(
                    review_id=symptom + ':' + data['names'][i], symptom=symptom, error_type=kind,
                    label_file_name=data['names'][i], gold=int(y[i]), prediction=int(p[i]),
                    probability=float(data['probabilities'][i, j]), threshold=float(data['thresholds'][j]),
                    all_target_labels=json.dumps([s for s, v in zip(LABELS, data['truth'][i]) if v], ensure_ascii=False),
                    transcript=data['texts'][i], review_category='', evidence='', review_note='',
                ))
    # Mix error/control order so blind inspection does not reveal the group.
    random.Random(seed).shuffle(records)
    return records


def write_csv(path, rows):
    path = Path(path)
    require(bool(rows), '저장할 행이 없습니다.')
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def prepare_review(data, output_root, reviewer='reviewer1', focus=('오심',), per_group=20, seed=42):
    require(re.fullmatch(r'[A-Za-z0-9_-]+', reviewer) is not None, '검토자 이름은 영문·숫자·밑줄·하이픈만 사용하세요.')
    samples = make_samples(data, focus, per_group, seed)
    settings = dict(signature=data['signature'], cache_sha256=data['cache_sha256'],
                    text_sha256=data['text_sha256'], focus=list(focus), per_group=per_group, seed=seed)
    key = sha(json.dumps(settings, sort_keys=True).encode())[:12]
    directory = Path(output_root) / key
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f'review_{reviewer}.csv'
    if path.exists():
        with path.open(encoding='utf-8-sig', newline='') as f:
            existing = list(csv.DictReader(f))
        require([r['review_id'] for r in existing] == [r['review_id'] for r in samples],
                '기존 검토 파일의 사례 목록이 달라 덮어쓰지 않습니다.')
        for new, old in zip(samples, existing):
            require(old.get('review_category', '') in ('', *CATEGORIES),
                    '기존 검토 CSV의 분류 코드가 올바르지 않습니다. 파일을 유지하고 중단합니다.')
            for field in ('review_category', 'evidence', 'review_note'):
                new[field] = old[field]
    else:
        write_csv(path, samples)
    write_csv(directory / 'symptom_counts.csv', confusion_rows(data))
    cooccurrence = []
    nausea, vomiting = data['truth'][:, 6], data['truth'][:, 1]
    for n, v, label in [(1, 0, '오심만'), (0, 1, '구토만'), (1, 1, '둘 다'), (0, 0, '둘 다 없음')]:
        mask = (nausea == n) & (vomiting == v)
        cooccurrence.append(dict(group=label, samples=int(mask.sum()),
                                 predicted_nausea=int(data['predictions'][mask, 6].sum())))
    write_csv(directory / 'nausea_vomiting_counts.csv', cooccurrence)
    (directory / 'analysis_info.json').write_text(json.dumps(
        {**settings, 'scope': SCOPE, 'samples': len(data['names']), 'review_samples': len(samples),
         'labels_modified': False, 'automatic_cause_assignment': False}, ensure_ascii=False, indent=2), encoding='utf-8')
    return path, samples


def save_review(path, rows, index, category, evidence='', note=''):
    require(category in CATEGORIES, '분류를 선택하세요.')
    require(bool(evidence.strip() or note.strip()), '근거 문장 또는 판단 메모를 적어주세요.')
    require(type(index) is int and 0 <= index < len(rows), '사례 번호를 확인하세요.')
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        saved = list(csv.DictReader(f))
    fields = ('review_id', 'review_category', 'evidence', 'review_note')
    require(len(saved) == len(rows) and all(
        all(a.get(field) == b.get(field) for field in fields)
        for a, b in zip(saved, rows)),
        '다른 탭 또는 파일에서 검토 내용이 바뀌었습니다. 분석 셀과 검토 화면을 다시 실행해 불러오세요.')
    updated = [dict(row) for row in rows]
    updated[index].update(review_category=category, evidence=evidence.strip(), review_note=note.strip())
    write_csv(path, updated)
    rows[index].update(updated[index])


def review_summary(rows):
    result = []
    for symptom in dict.fromkeys(row['symptom'] for row in rows):
        for kind in ('FP', 'FN', 'TP', 'TN'):
            group = [r for r in rows if r['symptom'] == symptom and r['error_type'] == kind]
            counts = Counter(r['review_category'] for r in group if r['review_category'])
            result.append(dict(symptom=symptom, group=kind, sampled=len(group), reviewed=sum(counts.values()),
                               **{code: counts[code] for code in CATEGORIES}))
    return result


In [ ]:
print("ZIP에서 Training 전사와 저장된 확률을 읽습니다. Drive 속도에 따라 잠시 걸릴 수 있습니다.", flush=True)
data = load_analysis(INPUT_ZIP, VER9_DIR)
review_path, review_rows = prepare_review(data, OUTPUT_ROOT, REVIEWER, FOCUS, PER_GROUP, SEED)
counts = confusion_rows(data)
print(f"Training 보정 집합: {len(data['names']):,}건")
print("증상       TP     FP     FN     TN   정밀도   재현율")
for row in counts:
    print(f"{row['symptom']:6} {row['TP']:5} {row['FP']:6} {row['FN']:6} {row['TN']:6}  {row['precision']:.3f}  {row['recall']:.3f}")
print("위 숫자는 기존 보정 집합에서 확정 임계값을 적용한 기술 통계입니다.")
print("검토 파일:", review_path)
print("검토 표본:", len(review_rows), "건 / 작성 완료:", sum(bool(r['review_category']) for r in review_rows))
print("오심·구토 정답 조합:")
with (review_path.parent / "nausea_vomiting_counts.csv").open(encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        print(row)


## 어떤 기준으로 분류하나요?

먼저 정답을 숨긴 상태에서 전사를 읽고 증상 근거를 찾아보세요. 그다음 **정답·예측 보기**를 켭니다.

| 분류 | 기록할 내용 |
|---|---|
| 명확한 근거를 모델이 놓침 | 실제 증상이라고 판단한 문장과 앞뒤 문맥 |
| 질문·부정·과거 문맥을 혼동 | 질문과 대답을 함께 인용; 단어 하나만 보고 판단하지 않기 |
| 다른 증상과 혼동한 것으로 의심 | 구토 등 다른 증상 표현과 오심 판단의 차이; 모델 원인 확정은 아님 |
| 전사만으로 정답 근거를 판단하기 어려움 | 전부 읽어도 판단이 어려운 이유. 라벨 오류 확정과 구분 |
| 맞힌 사례이며 근거도 명확함 | TP/TN 대조군에서 모델이 올바르게 구분한 근거 |
| 기타 | 위 기준에 맞지 않는 이유 |

근거가 없으면 문장을 만들어 넣지 말고 메모에 “전체 전사에서 근거를 찾기 어려움”이라고 적으세요.
‘오심 단어 없음’만으로 정답을 부정하지 않습니다. 모르는 표현, 간접 표현, 짧은 대답도 앞뒤 문맥과 함께 봅니다.
검토 결과는 원본 라벨이나 학습 자료에 자동 반영하지 않습니다. 파일 이름은 사례 연결용이며 모델 특징으로 사용하지 않습니다.


In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display, clear_output

pending = next((i for i, r in enumerate(review_rows) if not r["review_category"]), 0)
case = widgets.BoundedIntText(value=pending + 1, min=1, max=len(review_rows), description="사례 번호")
reveal = widgets.Checkbox(value=False, description="정답·예측 보기")
category = widgets.Dropdown(options=[("분류를 선택하세요", "")] + [(v, k) for k, v in CATEGORIES.items()], layout=widgets.Layout(width="95%"))
evidence = widgets.Textarea(description="근거 문장", layout=widgets.Layout(width="95%", height="90px"))
note = widgets.Textarea(description="판단 메모", layout=widgets.Layout(width="95%", height="90px"))
body, details, status = widgets.HTML(), widgets.HTML(), widgets.HTML()
previous = widgets.Button(description="이전")
next_case = widgets.Button(description="다음 / 건너뛰기")
save = widgets.Button(description="저장하고 다음", button_style="success")

def draw_details(*_):
    r = review_rows[case.value - 1]
    if reveal.value:
        details.value = (f"<b>{html.escape(r['error_type'])}</b> | 정답={r['gold']}, 예측={r['prediction']} "
                         f"| 확률={float(r['probability']):.4f}, 임계값={float(r['threshold']):.4f}<br>"
                         f"전체 정답: {html.escape(r['all_target_labels'])}")
    else:
        details.value = "정답·예측은 숨겨져 있습니다. 전사에서 근거를 먼저 찾아보세요."

def draw_case(*_):
    r = review_rows[case.value - 1]
    done = sum(bool(x['review_category']) for x in review_rows)
    body.value = (f"<b>{case.value}/{len(review_rows)} · {html.escape(r['symptom'])} · 검토 완료 {done}건</b>"
                  f"<pre style='white-space:pre-wrap;max-height:480px;overflow:auto;border:1px solid #ccc;padding:12px'>{html.escape(r['transcript'])}</pre>")
    category.value = r['review_category']
    evidence.value, note.value = r['evidence'], r['review_note']
    reveal.value = False
    draw_details()

case.observe(draw_case, names="value")
reveal.observe(draw_details, names="value")
previous.on_click(lambda _: setattr(case, "value", max(1, case.value - 1)))
next_case.on_click(lambda _: setattr(case, "value", min(len(review_rows), case.value + 1)))

def save_and_next(_):
    index = case.value - 1
    try:
        save_review(review_path, review_rows, index, category.value, evidence.value, note.value)
        write_csv(review_path.parent / f"review_summary_{REVIEWER}.csv", review_summary(review_rows))
    except Exception as exc:
        status.value = "<b>저장 실패:</b> " + html.escape(str(exc))
        return
    status.value = "Drive 저장 완료: " + html.escape(str(review_path))
    remaining = [i for i in range(index + 1, len(review_rows)) if not review_rows[i]['review_category']]
    remaining += [i for i in range(index) if not review_rows[i]['review_category']]
    if remaining:
        case.value = remaining[0] + 1
    else:
        draw_case()
        status.value += "<br><b>모든 사례 검토 완료. 마지막 집계 셀을 다시 실행하세요.</b>"

save.on_click(save_and_next)
draw_case()
display(widgets.VBox([case, body, reveal, details, category, evidence, note,
                       widgets.HBox([previous, next_case, save]), status]))
print("이전/다음 버튼은 미저장 입력을 저장하지 않습니다. 작성 후에는 '저장하고 다음'을 누르세요.")


## 검토 후 무엇을 보면 되나요?

아래 셀은 검토를 마친 뒤 다시 실행하세요. `FP`와 `FN`을 **따로** 비교합니다.

- **명확한 근거를 놓친 FN이 많음:** 표현과 문맥 처리 개선의 후보입니다. 해당 표현을 포함한 학습 사례와 대조합니다.
- **질문·부정·과거 혼동이 많음:** 전후 발화 연결을 유지하는 입력 구성을 같은 모델에서 비교할 근거가 됩니다.
- **다른 증상 혼동 FP가 많음:** 오심만/구토만/둘 다인 Training 사례의 표현 차이를 더 읽어봅니다.
- **전사만으로 판단 어려운 FN이 많음:** 대표 원본 사례와 함께 라벨 부여 기준을 문의합니다. 자동 삭제·재라벨링하지 않습니다.
- TP/TN은 같은 표현을 맞힌 이유를 비교하는 대조군입니다. 여러 문제가 있으면 주된 유형 하나를 고르고 나머지는 메모합니다.

이 소수 표본으로 전체 라벨 오류율이나 모델 성능 상한을 결론내리지 마세요.
한 사람이 작성한 분류는 가설입니다. 다른 검토자의 독립 결과와 의견이 다른 사례부터 합의하면 좋습니다.
다음 학습 실험은 확인된 가설 하나만 바꾸고, 학습에서 제외한 데이터에서 비교합니다.


In [ ]:
with review_path.open(encoding="utf-8-sig", newline="") as f:
    saved_rows = list(csv.DictReader(f))
summary = review_summary(saved_rows)
write_csv(review_path.parent / f"review_summary_{REVIEWER}.csv", summary)
for row in summary:
    print(f"\n{row['symptom']} {row['group']}: 검토 {row['reviewed']}/{row['sampled']}건")
    for code, label in CATEGORIES.items():
        if row[code]:
            print(f"  {label}: {row[code]}건")
print("\n결과 폴더:", review_path.parent)
print("review_검토자.csv: 근거 문장과 판단 메모")
print("review_summary_검토자.csv: 직접 작성한 분류 집계")
print("symptom_counts.csv: 9개 증상의 전체 보정 집합 TP/FP/FN/TN")
print("nausea_vomiting_counts.csv: 전체 보정 집합의 오심·구토 조합")
